# Function Calling, Under the Hood

Every agent framework hides the same loop: send messages to the model, get back a structured request to call a function, run that function yourself, feed the result back, repeat. Most walkthroughs skip straight to a framework doing this for you, which is convenient, but it leaves people thinking the model somehow runs code.

It does not. This notebook builds that loop by hand, one piece at a time, so you can see exactly what crosses the wire and who is doing what.

What we look at, in order:
- the actual JSON schema the model receives for a tool, not just the Python function
- the raw response the model returns, before anything gets executed
- the full manual loop: call model, execute tool, feed result back, repeat
- the same idea for a chain of dependent tool calls (sequential)
- the same idea when the model asks for several independent tool calls at once (parallel)

No agent abstraction anywhere in this notebook. `create_agent` and prebuilt agents are the subject of the next notebook, once you have seen what they are doing underneath.


## Setup

What: install the libraries we need.

Why: same reasoning as the HITL notebook, worth confirming these import cleanly on whatever version Colab hands you before relying on them.


In [1]:
!pip install -q langgraph langchain langchain-google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


## API key

What: load the Gemini key into the environment.

Why: `ChatGoogleGenerativeAI` reads it from the environment rather than us passing it around as a plain string.


In [3]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    import getpass
    if "GOOGLE_API_KEY" not in os.environ:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")

print("key loaded")


key loaded


## Part 1: Define the tools

What: one tool built on a Pydantic schema, plus a small set of tools for a sequential customer-support chain and a weather tool for the parallel demo later.

Why Pydantic: the docstring tells the model *when* to reach for a tool, the Pydantic model tells it *exactly what shape the arguments must be*. Field descriptions are not just documentation, the model reads them to decide what to put in each argument.


In [4]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field


class StockQuery(BaseModel):
    ticker: str = Field(description="Stock ticker symbol, e.g. GOOG")
    days: int = Field(default=7, description="Number of historical days to look back")


@tool(args_schema=StockQuery)
def get_stock_data(ticker: str, days: int = 7) -> str:
    """Fetches historical stock prices for a ticker."""
    # mock data, stands in for a real market data API call
    return f"Ticker: {ticker}, mean price over {days} days: $185.40"


@tool
def lookup_customer_id(name: str) -> str:
    """Look up a customer's account id from their full name."""
    return "CUST-4471"


@tool
def fetch_order_history(customer_id: str) -> str:
    """Fetch recent order history for a given customer id."""
    return "3 orders in the last 90 days, most recent total: $240.00"


@tool
def calculate_discount(customer_id: str, order_total: float) -> str:
    """Calculate a loyalty discount for a customer given an order total."""
    discount = round(order_total * 0.1, 2)
    return f"Discount for {customer_id}: ${discount}"


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # mock data, stands in for a real weather API call
    fake_data = {"Tokyo": "22C, clear", "London": "14C, rain", "New York": "18C, cloudy"}
    return fake_data.get(city, "no data for this city")


## Part 2: What the model actually receives

What: print the schema for `get_stock_data`, first as Pydantic sees it, then in the exact tool-calling format the model is given alongside your prompt.

Why: this is the part most walkthroughs skip. The model is never shown your Python function body, it is shown this schema. Everything it can possibly know about the tool - its name, its arguments, their types, their descriptions - comes from here and nowhere else.


In [5]:
import json
from langchain_core.utils.function_calling import convert_to_openai_tool

print("Pydantic's own view of the arguments:")
print(json.dumps(get_stock_data.args_schema.model_json_schema(), indent=2))

print()
print("the schema actually sent alongside your prompt:")
print(json.dumps(convert_to_openai_tool(get_stock_data), indent=2))


Pydantic's own view of the arguments:
{
  "properties": {
    "ticker": {
      "description": "Stock ticker symbol, e.g. GOOG",
      "title": "Ticker",
      "type": "string"
    },
    "days": {
      "default": 7,
      "description": "Number of historical days to look back",
      "title": "Days",
      "type": "integer"
    }
  },
  "required": [
    "ticker"
  ],
  "title": "StockQuery",
  "type": "object"
}

the schema actually sent alongside your prompt:
{
  "type": "function",
  "function": {
    "name": "get_stock_data",
    "description": "Fetches historical stock prices for a ticker.",
    "parameters": {
      "properties": {
        "ticker": {
          "description": "Stock ticker symbol, e.g. GOOG",
          "type": "string"
        },
        "days": {
          "default": 7,
          "description": "Number of historical days to look back",
          "type": "integer"
        }
      },
      "required": [
        "ticker"
      ],
      "type": "object"
    }
  }


This is the OpenAI-style tool schema, the shared format LangChain builds before translating it into whatever shape a specific provider wants. Gemini gets the same information, just repackaged into its own function-declaration format on the way out. The content is what matters here, not the wrapper.


## Part 3: Ask the model, look at the raw answer

What: bind the tool to Gemini, send a prompt, print the response exactly as it comes back, before we do anything with it.

Why: watch `response.content`, it will be empty or close to it. The model is not answering the question yet, it is asking us to run a function and come back with the result. `response.tool_calls` is that request in structured form: which function, with which arguments, tagged with an id so we know which result belongs to which call later.


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm_with_tools = llm.bind_tools([get_stock_data])

response = llm_with_tools.invoke("Check GOOG stock for the last 5 days")

print("response.content:", repr(response.content))
print()
print("response.tool_calls:")
for call in response.tool_calls:
    print(f"  function: {call['name']}")
    print(f"  arguments: {call['args']}")
    print(f"  call id: {call['id']}")


response.content: ''

response.tool_calls:
  function: get_stock_data
  arguments: {'days': 5, 'ticker': 'GOOG'}
  call id: f6ba3e3f-5cd6-432d-b634-e21f5a73cf5e


## Part 4: Closing the loop by hand

What: take that tool call, actually run the matching Python function, wrap the result the way the model expects, send it back, and get the real final answer.

Why: this is the entire trick behind every "agent". Nothing hidden happens between Part 3 and here, it is a dictionary lookup and a function call. `create_agent` in the next notebook does exactly these steps for you, on a loop.


In [7]:
from langchain_core.messages import HumanMessage, ToolMessage

tools_by_name = {"get_stock_data": get_stock_data}

messages = [HumanMessage("Check GOOG stock for the last 5 days")]
response = llm_with_tools.invoke(messages)
messages.append(response)

for call in response.tool_calls:
    tool_fn = tools_by_name[call["name"]]
    result = tool_fn.invoke(call["args"])
    print(f"ran {call['name']}({call['args']}) -> {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

final = llm_with_tools.invoke(messages)
print()
print("final answer:", final.content)


ran get_stock_data({'days': 5, 'ticker': 'GOOG'}) -> Ticker: GOOG, mean price over 5 days: $185.40

final answer: The mean price for GOOG over the last 5 days is $185.40.


In [8]:
messages

[HumanMessage(content='Check GOOG stock for the last 5 days', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_stock_data', 'arguments': '{"days": 5, "ticker": "GOOG"}'}, '__gemini_function_call_thought_signatures__': {'d72bdb7d-90f9-4012-a482-39b0b32e0b56': 'CqwCAWkUfRPxayxVdT/f4LLHWFe/nitRTeYMelT5Sl5lvdD2nAVH6V4s57pMOhwWGlAQIX1n3DwqNATQtqede8kpG/p5sa84w2NWaYDz1VNHsncz3OetQksOy2hGOmMYbq0/fnejtfVtd1jabBnWDum1CU/5a0fnasNKj5Bk84TssjhlLJteImzWhG2FME26RPtih6XNUHVDF/d1DzPQtDAk4X3qiw4Qtwj5z3SVD46AjazbyWAW9B0SmvxbH90TQJCra4Bl9leqs7UWnTXS1H2FNpjbYGlBW4nTHyxTpHv9iLpRgzXLPcGI6GfxZgfw+YP7W1mulfTz7gMKM2IooEqGJOKGknMC/SUM5zJ8D0wxsP+UlMmQxqiplfYcL1oAppVvA5hQbUWutEMiqIQ0'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d1e4-ce46-7bf3-aa12-e74107cf801d-0', tool_calls=[{'name': 'get_stock_data', 'args': {'days': 5, 'ticker': 'GOOG

## Part 5: Sequential tool chaining, step by step

What: a three-step chain where each step needs the previous one's result - look up a customer id, fetch their orders, calculate a discount on the total. We drive this with a loop rather than hardcoding three calls, since we don't know upfront how many turns the model will need.

Why this has to be sequential: `calculate_discount` needs an order total that only exists after `fetch_order_history` has run, which needs a customer id that only exists after `lookup_customer_id` has run. The model has to see each result before it can decide the next step, so this cannot be made parallel no matter how the code is written.


In [9]:
sequential_tools = [lookup_customer_id, fetch_order_history, calculate_discount]
tools_by_name.update({t.name: t for t in sequential_tools})
llm_with_sequential_tools = llm.bind_tools(sequential_tools)

messages = [HumanMessage(
    "Find the customer id for Priya Shah, look up her order history, "
    "then calculate her loyalty discount on a $240 order."
)]

step = 1
while True:
    response = llm_with_sequential_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        print(f"\nfinal answer: {response.content}")
        break

    for call in response.tool_calls:
        result = tools_by_name[call["name"]].invoke(call["args"])
        print(f"step {step}: {call['name']}({call['args']}) -> {result}")
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        step += 1


step 1: lookup_customer_id({'name': 'Priya Shah'}) -> CUST-4471
step 2: fetch_order_history({'customer_id': 'CUST-4471'}) -> 3 orders in the last 90 days, most recent total: $240.00
step 3: calculate_discount({'order_total': 240, 'customer_id': 'CUST-4471'}) -> Discount for CUST-4471: $24.0

final answer: [{'type': 'text', 'text': "Priya Shah's customer ID is CUST-4471. She has made 3 orders in the last 90 days, with the most recent total being $240. Her loyalty discount on a $240 order is $24.0.", 'extras': {'signature': 'CogCAWkUfRNZe9jLhi5EJMZmQN4VrQxKwUQe1geA6H8/8Dpf44QWzEHkDYCltdlagVLaKwBKvc2YCsD0rSsC3KMqUGYWLZVnI55aNkjROjdxLZj1972uuwAiU/HOnMwnLN7Wxv8MoWclgQ+25980f+04rIJ+MLWVegRA0UvlyxpJtBJ9yWbZJUMN7hHgx4r3HVYjr26BJY87CqqwKq+a3Mwno5JUIcfOPIEXLimBkjHY51NX7X1VNf/hAckPPKIjF16ugWJQlrwrov2Lk8xZIKiagD9scxS6XX05iOJaWLUov1vGK+et5P20kM8zLW+giR8K4OTvsCp71fHrn0RFHGzhONfU8mFlkdGW'}}]


## Part 6: Parallel tool calling

What: ask about three independent cities in one prompt and see whether Gemini asks for all three tool calls in a single turn. Since none of these calls depend on each other, we run them concurrently instead of one after another.

Why it's faster: three sequential calls means three round trips through the tool, one after another. If the model batches them into one turn, we can fire all three at once and wait for the slowest one instead of the sum of all three.


In [10]:
llm_with_weather = llm.bind_tools([get_weather])

response = llm_with_weather.invoke(
    "Compare the weather in Tokyo, London, and New York."
)

print(f"model asked for {len(response.tool_calls)} tool call(s) in this turn:")
for call in response.tool_calls:
    print(" ", call["name"], call["args"])


model asked for 3 tool call(s) in this turn:
  get_weather {'city': 'Tokyo'}
  get_weather {'city': 'London'}
  get_weather {'city': 'New York'}


If you see fewer than 3 calls, try rephrasing to make it more explicit that all three need comparing together, models vary in how eagerly they batch calls, this is worth trying live with the group.

Either way, here is the actual latency difference once the calls are in hand, using a deliberately slow mock tool to make the gap obvious.


In [11]:
import time
from concurrent.futures import ThreadPoolExecutor


def slow_get_weather(city):
    time.sleep(1)  # stands in for real network latency
    return get_weather.invoke({"city": city})


cities = ["Tokyo", "London", "New York"]

t0 = time.time()
sequential_results = [slow_get_weather(c) for c in cities]
t1 = time.time()
print(f"sequential: {round(t1 - t0, 2)}s -> {sequential_results}")

t0 = time.time()
with ThreadPoolExecutor() as pool:
    parallel_results = list(pool.map(slow_get_weather, cities))
t1 = time.time()
print(f"parallel:   {round(t1 - t0, 2)}s -> {parallel_results}")


sequential: 3.0s -> ['22C, clear', '14C, rain', '18C, cloudy']
parallel:   1.0s -> ['22C, clear', '14C, rain', '18C, cloudy']


## Recap

Reading the schema, sending the messages, matching a tool call to a function, running it, sending the result back, looping until there is a final answer, running independent calls concurrently - all of it is exactly what `create_agent` does internally. The next notebook opens that box up and compares building on top of it against building the graph yourself.
